# 02 · Training and predicting

Notebook 01 showed how calibration (training) works. Here we load a **fully pre-trained** model that ships with the repo and use it to **predict** — i.e. produce a probabilistic streamflow ensemble.

The bundled model [`examples/GPU_HYPE_2026-03-24_13h15.pkl`](../examples/GPU_HYPE_2026-03-24_13h15.pkl) is a `GPU_PSO` with:
- a `HYPE` model on the Türkheim catchment (outlet `50675`),
- a **250-member** calibrated population over 17 parameters,
- calibrated on **1980–1999** (validation starts 2000).

We forecast the **validation window 2000–2004**, which the model never saw during calibration.

In [ ]:
# --- Bootstrap: run from the repo root ---
import os, sys
from pathlib import Path

def find_repo_root(start=None):
    start = start or Path.cwd()
    for p in [start, *start.parents]:
        if (p / "gpu.py").exists():
            return p
    raise RuntimeError("Could not find the repo root above %s" % start)

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
%matplotlib inline

from gpu import GPU
from functions import processBands
print("Imports OK")

## 1 · Load the trained model

`GPU.load` restores the whole optimiser (model + error model + the calibrated population). Because the pickle stores a *relative* HYPE-folder path, we re-point it at the bundled folder.

In [ ]:
model = GPU.load("examples/GPU_HYPE_2026-03-24_13h15.pkl")
model.modelObject.DefaultFolderLoc = Path("examples/set7_germany_tuerkheim")

bands = model.opt["bands"]
print("trained:", model.trained)
print("population (members \u00d7 variables):", model.population.shape)
print("calibrated parameters:", model.modelObject.calibration_parameters)
print("probability bands:", bands)

## 2 · Run the ensemble forecast

For a HYPE model, `predict()` re-runs the executable with **every member** of the calibrated population over the requested window and aggregates the simulations into probability bands.

A loaded model has no temporary run folders, so we (1) re-create them, (2) set the forecast window, and (3) set the matching `records` index before predicting.

*(250 members over 5 years across parallel HYPE runs — expect a few minutes.)*

In [ ]:
init_date = pd.to_datetime("2000-01-01")
final_date = pd.to_datetime("2004-12-31")
records = pd.date_range(init_date, final_date, freq="D")

model.modelObject.init_tmpFiles()
model.modelObject.set_simulation_Dates(init_date, final_date, b_init=init_date)
model.modelObject.records = records

aggregated = model.predict()                 # (n_steps, n_bands)
aggregated = processBands(aggregated, bands)  # enforce monotone bands, fill gaps
model.modelObject.remove_tmpFiles()
print("aggregated bands shape:", aggregated.shape)

## 3 · Assemble the forecast table

We wrap the band array in a `DataFrame` with one column per probability level (a single `0D` leadtime), and save it for the results notebook.

In [ ]:
columns = pd.MultiIndex.from_product(
    [[pd.Timedelta("0D")], bands], names=["Leadtime", "Probability"]
)
predictions = pd.DataFrame(aggregated, index=records, columns=columns)
predictions.to_csv("examples/predictions.csv")
print("saved \u2192 examples/predictions.csv")
predictions.iloc[:3, :5]

## 4 · A first look at the ensemble

Shaded bands are predictive quantile intervals; the line is the ensemble median, overlaid on the observations.

In [ ]:
obs = pd.read_csv("examples/Qobs.txt", sep="\t", header=0,
                  index_col="Date", parse_dates=True)["50675"]
obs_win = obs.reindex(records)

band_arr = predictions.values            # (n_steps, n_bands)
def band_col(level):
    return band_arr[:, int(np.argmin(np.abs(np.array(bands) - level)))]

median = 0.5 * (band_col(0.45) + band_col(0.55))

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(records, band_col(0.05), band_col(0.95), color="tab:blue", alpha=0.20, label="5\u201395%")
ax.fill_between(records, band_col(0.25), band_col(0.75), color="tab:blue", alpha=0.35, label="25\u201375%")
ax.plot(records, median, color="tab:blue", lw=1.0, label="Ensemble median")
ax.plot(obs_win.index, obs_win.values, color="black", lw=0.8, label="Observed")
ax.set_ylabel("Discharge [m\u00b3/s]"); ax.set_title("Probabilistic forecast \u2014 validation window")
ax.legend(ncol=4, fontsize=9)
plt.tight_layout()

## Next

[**03 · Results**](03_results.ipynb) takes this forecast and evaluates it: hydrograph diagnostics, the reliability (PIT / Q-Q) diagram, and skill metrics against the observations and HYPE's own calibration.